In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")


def import_data(csv_file):
    df = pd.read_csv(csv_file)
    patient_ids = df.iloc[:, 0].astype(str).str.strip()
    features = df.iloc[:, 1:316]  # Adjust as needed

    feats = []
    unique_ids = patient_ids.unique()
    for pid in unique_ids:
        patient_data = features[patient_ids == pid].values
        feats.append(patient_data)
    return feats, unique_ids, features.columns.tolist()


def import_labels(csv_file, label_column_name):
    df = pd.read_csv(csv_file)
    patient_ids = df.iloc[:, 0].astype(str).str.strip()
    labels_df = df[[df.columns[0], label_column_name]].copy()
    labels_df[df.columns[0]] = labels_df[df.columns[0]].astype(str).str.strip()
    labels_df = labels_df.drop_duplicates(subset=df.columns[0]).set_index(df.columns[0])
    unique_ids = patient_ids.unique()
    labels = labels_df.loc[unique_ids][label_column_name].values
    return labels.astype(np.int64)

def import_proteomics(proteomics_csv):
    df = pd.read_csv(proteomics_csv)
    df[df.columns[0]] = df[df.columns[0]].astype(str).str.strip()
    df = df.set_index(df.columns[0])

    # Replace invalid entries with NaN and convert to float
    df = df.replace('#VALUE!', np.nan)
    df = df.astype(float)

    # Optionally impute missing values with column means
    df = df.fillna(df.mean())

    return df



def import_covariates(covariates_csv):
    df = pd.read_csv(covariates_csv)
    df[df.columns[0]] = df[df.columns[0]].astype(str).str.strip()
    df = df.set_index(df.columns[0])
    return df


def regress_out(Z, M):
    model = LinearRegression().fit(Z, M)
    predictions = model.predict(Z)
    residuals = M - predictions
    return residuals


def aggregate_patient_features(feats, method="mean"):
    agg_feats = []
    for patient in feats:
        if method == "mean":
            agg_feats.append(np.mean(patient, axis=0))
        elif method == "max":
            agg_feats.append(np.max(patient, axis=0))
        elif method == "min":
            agg_feats.append(np.min(patient, axis=0))
        else:
            raise ValueError("Unsupported aggregation method.")
    return np.array(agg_feats)


def save_results_csv(true_labels, pred_labels, pred_probs, filename):
    df = pd.DataFrame({
        "TrueLabel": true_labels,
        "PredLabel": pred_labels,
        "PredProb": pred_probs
    })
    df.to_csv(filename, index=False)
    print(f"Saved predictions to {filename}")


def plot_and_save_confusion_matrix(true_labels, pred_labels, filename):
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0, 1], yticklabels=[0, 1])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.savefig(filename)
    plt.close()
    print(f"Saved confusion matrix to {filename}")


def plot_and_save_roc_curve(true_labels, pred_probs, filename):
    fpr, tpr, _ = roc_curve(true_labels, pred_probs)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label="ROC Curve")
    plt.plot([0, 1], [0, 1], "k--", label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.savefig(filename)
    plt.close()
    print(f"Saved ROC curve to {filename}")


def spearman_corr_matrix(X, Y):
    from scipy.stats import rankdata

    X_ranked = np.apply_along_axis(rankdata, 0, X)
    Y_ranked = np.apply_along_axis(rankdata, 0, Y)

    X_ranked_c = X_ranked - X_ranked.mean(axis=0)
    Y_ranked_c = Y_ranked - Y_ranked.mean(axis=0)

    X_norm = X_ranked_c / np.linalg.norm(X_ranked_c, axis=0)
    Y_norm = Y_ranked_c / np.linalg.norm(Y_ranked_c, axis=0)

    corr_matrix = np.dot(X_norm.T, Y_norm)
    return corr_matrix



In [2]:

def cross_validate_xgboost_with_feature_selection(
        csv_file,
        proteomics_file,
        covariates_file,
        label_column,
        folds=10,
        agg_method="mean",
        save_directory="./",
        num_est=100,
        seed=None,
        top_features_num=50,
        prob_threshold=0.35,
):
    print("Loading data...")
    feats, unique_ids, feature_names = import_data(csv_file)
    labels = import_labels(csv_file, label_column)
    proteomics_df = import_proteomics(proteomics_file)
    covariates_df = import_covariates(covariates_file)

    print(f"Total patients: {len(labels)}")
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    all_true = []
    all_pred = []
    all_prob = []

    os.makedirs(save_directory, exist_ok=True)

    for fold, (train_idx, test_idx) in enumerate(skf.split(feats, labels), 1):
        print(f"\nFold {fold}/{folds}")

        feats_train = [feats[i] for i in train_idx]
        feats_test = [feats[i] for i in test_idx]
        y_train = labels[train_idx]
        y_test = labels[test_idx]

        train_ids = unique_ids[train_idx]
        test_ids = unique_ids[test_idx]

        proteomics_train = proteomics_df.loc[train_ids].dropna(axis=1)
        proteomics_test = proteomics_df.loc[test_ids][proteomics_train.columns]

        covariates_train = covariates_df.loc[train_ids]
        covariates_test = covariates_df.loc[test_ids]

        all_train = np.vstack(feats_train)
        scaler = StandardScaler().fit(all_train)
        feats_train_scaled = [scaler.transform(p) for p in feats_train]
        feats_test_scaled = [scaler.transform(p) for p in feats_test]

        X_train_full = aggregate_patient_features(feats_train_scaled, method=agg_method)
        X_test_full = aggregate_patient_features(feats_test_scaled, method=agg_method)

        # Regress out covariates
        X_train_resid = regress_out(covariates_train.values, X_train_full)
        Y_train_resid = regress_out(covariates_train.values, proteomics_train.values)

        corr_matrix = spearman_corr_matrix(X_train_resid, Y_train_resid)
        corr_matrix = np.nan_to_num(corr_matrix)
        mean_abs_corr = np.mean(np.abs(corr_matrix), axis=1)

        fold_dir = os.path.join(save_directory, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rank_df = pd.DataFrame({
            'glom_feature_index': np.arange(X_train_full.shape[1]),
            'mean_abs_spearman_corr': mean_abs_corr
        })
        rank_df.to_csv(os.path.join(fold_dir, "feature_correlations.csv"), index=False)
        print(f"Saved feature correlations to {fold_dir}/feature_correlations.csv")

        top_features_indices = np.argsort(-mean_abs_corr)[:top_features_num]
        top_feature_names = [feature_names[i] for i in top_features_indices]
        with open(os.path.join(fold_dir, "selected_feature_names.txt"), "w") as f:
            for name in top_feature_names:
                f.write(name + "\n")
        print(f"Saved top feature names to {fold_dir}/selected_feature_names.txt")

        X_train = X_train_full[:, top_features_indices]
        X_test = X_test_full[:, top_features_indices]

        model = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            use_label_encoder=False,
            scale_pos_weight=(sum(y_train == 0) / sum(y_train == 1)),
            max_depth=4,
            learning_rate=0.2,
            n_estimators=num_est,
            verbosity=0,
            random_state=42
        )

        model.fit(X_train, y_train)
        prob = model.predict_proba(X_test)[:, 1]
        pred = (prob >= prob_threshold).astype(int)

        all_true.extend(y_test)
        all_pred.extend(pred)
        all_prob.extend(prob)

        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)
        mcc = matthews_corrcoef(y_test, pred)

        print(f"Fold {fold} Acc: {acc:.4f} | AUC: {auc:.4f} | MCC: {mcc:.4f}")

    classification_rep = classification_report(all_true, all_pred, digits=4)
    overall_acc = accuracy_score(all_true, all_pred)
    overall_auc = roc_auc_score(all_true, all_prob)
    overall_mcc = matthews_corrcoef(all_true, all_pred)
    tn, fp, fn, tp = confusion_matrix(all_true, all_pred).ravel()
    specificity = tn / (tn + fp)

    with open(os.path.join(save_directory, "results_summary.txt"), "w") as f:
        f.write("Classification Report (Aggregated over all folds):\n")
        f.write(classification_rep)
        f.write("\n\n")
        f.write(f"Overall Accuracy: {overall_acc:.4f}\n")
        f.write(f"Overall AUC: {overall_auc:.4f}\n")
        f.write(f"Overall MCC: {overall_mcc:.4f}\n")
        f.write(f"Overall Specificity: {specificity:.4f}\n")

    save_results_csv(all_true, all_pred, all_prob, os.path.join(save_directory, "predictions.csv"))
    plot_and_save_confusion_matrix(all_true, all_pred, os.path.join(save_directory, "confusion_matrix.png"))
    plot_and_save_roc_curve(all_true, all_prob, os.path.join(save_directory, "roc_curve.png"))




In [3]:
# --- Run the pipeline ---
csv_file = "/blue/pinaki.sarder/narra.sr/5_DN_SKorea/data/combined_glom_features_with_outcomes.csv"
proteomics_file = "/blue/pinaki.sarder/narra.sr/5_DN_SKorea/data/normalized_proteomics_data.csv"
covariates_file = "/blue/pinaki.sarder/narra.sr/5_DN_SKorea/data/combined_patient_metadata.csv"
label = "2y_composite_outcome"
output_root = "./xgb_results_with_covariates"

cross_validate_xgboost_with_feature_selection(
    csv_file=csv_file,
    proteomics_file=proteomics_file,
    covariates_file=covariates_file,
    label_column=label,
    folds=10,
    agg_method="mean",
    save_directory=output_root,
    num_est=300,
    seed=42,
    top_features_num=80,
    prob_threshold=0.35
)


Loading data...
Total patients: 86

Fold 1/10
Saved feature correlations to ./xgb_results_with_covariates/fold_1/feature_correlations.csv
Saved top feature names to ./xgb_results_with_covariates/fold_1/selected_feature_names.txt
Fold 1 Acc: 0.5556 | AUC: 0.6667 | MCC: 0.0000

Fold 2/10
Saved feature correlations to ./xgb_results_with_covariates/fold_2/feature_correlations.csv
Saved top feature names to ./xgb_results_with_covariates/fold_2/selected_feature_names.txt
Fold 2 Acc: 0.6667 | AUC: 0.7222 | MCC: 0.5000

Fold 3/10
Saved feature correlations to ./xgb_results_with_covariates/fold_3/feature_correlations.csv
Saved top feature names to ./xgb_results_with_covariates/fold_3/selected_feature_names.txt
Fold 3 Acc: 0.7778 | AUC: 0.9444 | MCC: 0.5000

Fold 4/10
Saved feature correlations to ./xgb_results_with_covariates/fold_4/feature_correlations.csv
Saved top feature names to ./xgb_results_with_covariates/fold_4/selected_feature_names.txt
Fold 4 Acc: 0.7778 | AUC: 0.8889 | MCC: 0.5000

